# Quantization from Scratch on a Toy `make_moons` Dataset

> **Week 11 Lecture Demo** — CS 203: Software Tools and Techniques for AI · IIT Gandhinagar

In this notebook we will:

1. Generate a two-moons dataset **without sklearn** (just `numpy`).
2. Train a tiny MLP in PyTorch.
3. Implement **INT8 symmetric quantization from scratch** — no `torch.quantization`, no ONNX, no external libs. Just arithmetic.
4. Compare accuracy, memory, and decision boundary between the FP32 and INT8 models.

The whole point is that quantization is **not magic**: it is three lines of arithmetic per tensor. If you can do `round(x * 127 / max_abs)`, you can quantize a neural network.


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)


---

## 1. Build a two-moons dataset from scratch

`sklearn.datasets.make_moons` is just two interleaving half-circles. Let's rebuild it in four lines of NumPy so there is zero library magic.


In [ ]:
def make_moons(n_samples=400, noise=0.15, seed=0):
    """Generate a `make_moons`-style dataset using only numpy."""
    rng = np.random.default_rng(seed)
    n_a = n_samples // 2
    n_b = n_samples - n_a

    # Upper moon (class 0): half circle shifted left + up
    t_a = rng.uniform(0, math.pi, n_a)
    x_a = np.stack([np.cos(t_a), np.sin(t_a)], axis=1)

    # Lower moon (class 1): half circle shifted right + down
    t_b = rng.uniform(0, math.pi, n_b)
    x_b = np.stack([1 - np.cos(t_b), -np.sin(t_b) + 0.5], axis=1)

    X = np.concatenate([x_a, x_b], axis=0).astype(np.float32)
    y = np.concatenate([np.zeros(n_a), np.ones(n_b)], axis=0).astype(np.int64)

    # Add Gaussian noise
    X += rng.normal(scale=noise, size=X.shape).astype(np.float32)

    # Shuffle
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


X_np, y_np = make_moons(n_samples=1000, noise=0.20, seed=0)
print(f"X shape: {X_np.shape}, y shape: {y_np.shape}")
print(f"class balance: {np.bincount(y_np)}")


In [ ]:
# Quick look at the dataset
plt.figure(figsize=(5, 4))
plt.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], s=10, label="class 0")
plt.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], s=10, label="class 1")
plt.legend(); plt.title("Two-moons dataset"); plt.xlabel("x1"); plt.ylabel("x2")
plt.tight_layout(); plt.show()


---

## 2. Train a tiny MLP in FP32

2 inputs → 16 hidden → 16 hidden → 2 logits. Nothing fancy — this is the simplest thing that can get the moons right.


In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np)

# 80/20 train/test split
perm = torch.randperm(len(X))
n_tr = int(0.8 * len(X))
X_tr, y_tr = X[perm[:n_tr]], y[perm[:n_tr]]
X_te, y_te = X[perm[n_tr:]], y[perm[n_tr:]]

model = TinyMLP()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(400):
    opt.zero_grad()
    logits = model(X_tr)
    loss = F.cross_entropy(logits, y_tr)
    loss.backward()
    opt.step()
    if (epoch + 1) % 100 == 0:
        with torch.no_grad():
            acc = (model(X_te).argmax(1) == y_te).float().mean().item()
        print(f"epoch {epoch+1:4d}  loss={loss.item():.4f}  test_acc={acc:.4f}")

model.eval()
with torch.no_grad():
    fp32_acc = (model(X_te).argmax(1) == y_te).float().mean().item()
print(f"\nFP32 test accuracy: {fp32_acc:.4f}")


---

## 3. INT8 symmetric quantization — the core arithmetic

For every weight tensor `W`:

$$
\text{scale} = \frac{\max(|W|)}{127}, \qquad
q = \text{round}\!\left(\frac{W}{\text{scale}}\right),\qquad
\hat W = q \cdot \text{scale}
$$

- `q` is stored as **int8** (range `[-127, 127]`). One byte per parameter, 4× smaller than FP32.
- `scale` is one FP32 number per tensor.
- At inference we "dequantize" on the fly: `W_dequant = q * scale`, and then do the matmul in FP32.

Three lines. That's quantization.


In [ ]:
def quantize_tensor(x: torch.Tensor):
    """Symmetric per-tensor INT8 quantization.
    Returns: (int8_values, fp32_scale).
    """
    max_abs = x.abs().max().item()
    if max_abs == 0:
        return torch.zeros_like(x, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)
    return q, scale


def dequantize_tensor(q: torch.Tensor, scale: float) -> torch.Tensor:
    return q.to(torch.float32) * scale


# Sanity check: quantize a small tensor and see the round-trip error
w = torch.randn(4, 4)
q, s = quantize_tensor(w)
w_hat = dequantize_tensor(q, s)
print("original w:\n", w)
print("\nint8 q:\n", q)
print(f"\nscale: {s:.6f}")
print(f"max round-trip error: {(w - w_hat).abs().max().item():.6f}")


### Now quantize every `nn.Linear` in the trained model

We walk the three linear layers, quantize each weight tensor, and keep biases in FP32 (biases are tiny — quantizing them buys almost nothing and costs accuracy).


In [ ]:
def quantize_linear_layers(net: nn.Module):
    """Return a dict: layer_name -> (int8_weight, fp32_scale, fp32_bias)."""
    qstate = {}
    for name, module in net.named_modules():
        if isinstance(module, nn.Linear):
            q, s = quantize_tensor(module.weight.data)
            qstate[name] = {
                "q_weight": q,
                "scale": s,
                "bias": module.bias.data.clone() if module.bias is not None else None,
            }
    return qstate


qstate = quantize_linear_layers(model)
for name, entry in qstate.items():
    q = entry["q_weight"]
    print(f"{name:4s}  shape={tuple(q.shape)}  scale={entry['scale']:.6f}  "
          f"q_range=[{q.min().item()}, {q.max().item()}]")


### Run a forward pass with the quantized weights

We dequantize on the fly and run the rest of the computation in FP32. This is sometimes called **fake quantization** — a faithful simulation of what INT8 inference would produce, without needing INT8 matmul kernels.


In [ ]:
@torch.no_grad()
def forward_quantized(x, qstate):
    # fc1
    w = dequantize_tensor(qstate["fc1"]["q_weight"], qstate["fc1"]["scale"])
    x = F.relu(F.linear(x, w, qstate["fc1"]["bias"]))
    # fc2
    w = dequantize_tensor(qstate["fc2"]["q_weight"], qstate["fc2"]["scale"])
    x = F.relu(F.linear(x, w, qstate["fc2"]["bias"]))
    # fc3
    w = dequantize_tensor(qstate["fc3"]["q_weight"], qstate["fc3"]["scale"])
    x = F.linear(x, w, qstate["fc3"]["bias"])
    return x


with torch.no_grad():
    int8_logits = forward_quantized(X_te, qstate)
    int8_acc = (int8_logits.argmax(1) == y_te).float().mean().item()

print(f"FP32 test accuracy: {fp32_acc:.4f}")
print(f"INT8 test accuracy: {int8_acc:.4f}")
print(f"difference: {abs(fp32_acc - int8_acc):.4f}")


---

## 4. Memory footprint

Count the bytes. For each `nn.Linear`, the FP32 model stores weights in 4 bytes/param; the INT8 model stores weights in 1 byte/param plus one FP32 scale (4 bytes total per layer).


In [ ]:
def count_bytes_fp32(net):
    total = 0
    for p in net.parameters():
        total += p.numel() * 4  # FP32 = 4 bytes
    return total


def count_bytes_int8(qstate, net):
    total = 0
    # Quantized weights: 1 byte each, plus one FP32 scale per layer
    for name, entry in qstate.items():
        total += entry["q_weight"].numel() * 1  # int8 weights
        total += 4                                # FP32 scale
        if entry["bias"] is not None:
            total += entry["bias"].numel() * 4   # biases kept FP32
    return total


fp32_bytes = count_bytes_fp32(model)
int8_bytes = count_bytes_int8(qstate, model)

print(f"FP32 model size: {fp32_bytes} bytes")
print(f"INT8 model size: {int8_bytes} bytes")
print(f"compression ratio: {fp32_bytes / int8_bytes:.2f}x")


---

## 5. Decision boundary — side by side

Do the FP32 and INT8 models actually draw similar boundaries? Let's look.


In [ ]:
def decision_boundary(predict_fn, ax, title):
    x1_min, x1_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    x2_min, x2_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                           np.linspace(x2_min, x2_max, 200))
    grid = torch.from_numpy(np.stack([xx1.ravel(), xx2.ravel()], axis=1).astype(np.float32))
    with torch.no_grad():
        pred = predict_fn(grid).argmax(1).numpy().reshape(xx1.shape)
    ax.contourf(xx1, xx2, pred, alpha=0.30, levels=[-0.5, 0.5, 1.5])
    ax.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], s=8, label="class 0")
    ax.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], s=8, label="class 1")
    ax.set_title(title); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.legend(fontsize=7)


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
decision_boundary(lambda g: model(g),                       axes[0], f"FP32  (acc={fp32_acc:.3f})")
decision_boundary(lambda g: forward_quantized(g, qstate),   axes[1], f"INT8  (acc={int8_acc:.3f})")
plt.tight_layout(); plt.show()


---

## 6. Where does the quantization error show up?

Let's look at the **logit difference** between FP32 and INT8 over the full input grid. Red = model disagrees the most.


In [ ]:
with torch.no_grad():
    x1 = np.linspace(X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5, 200)
    x2 = np.linspace(X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5, 200)
    xx1, xx2 = np.meshgrid(x1, x2)
    grid = torch.from_numpy(np.stack([xx1.ravel(), xx2.ravel()], axis=1).astype(np.float32))
    fp32_logits = model(grid)
    int8_logits = forward_quantized(grid, qstate)
    diff = (fp32_logits - int8_logits).abs().max(dim=1).values.numpy().reshape(xx1.shape)

plt.figure(figsize=(5, 4))
cs = plt.contourf(xx1, xx2, diff, levels=20, cmap="Reds")
plt.colorbar(cs, label="max |FP32 - INT8| logit diff")
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=6, cmap="coolwarm", edgecolor="k", linewidth=0.2)
plt.title("Quantization error over input space"); plt.xlabel("x1"); plt.ylabel("x2")
plt.tight_layout(); plt.show()


---

## 7. Exercises

1. **Per-channel quantization.** Right now we use one scale for the whole weight tensor of each Linear. Extend `quantize_tensor` to take a `dim` argument and compute a separate scale per output channel (i.e. per row of the weight matrix). Does accuracy improve? By how much?

2. **Asymmetric quantization.** Add a zero-point `z` so the formula becomes `q = round(x / scale) + z` with `scale = (x_max - x_min) / 255` and output in `[0, 255]` (uint8). Implement `quantize_tensor_asym` / `dequantize_tensor_asym`. Compare accuracy to the symmetric version.

3. **Activation quantization.** We only quantized *weights*. In real INT8 inference, activations are also quantized (harder: you need calibration data to pick a scale). Implement static activation quantization: on a calibration pass, record `max|x|` after each ReLU, then in a quantized forward pass, quantize/dequantize the activations at those breakpoints. What happens to accuracy?

4. **Smaller bit-widths.** Change the clamp range from `[-127, 127]` to `[-7, 7]` (INT4). How much does accuracy degrade on this toy problem? When does it break?

5. **Quantize the biases too.** Keep biases in INT32 with the combined scale `scale_input * scale_weight`. Why is INT32 (not INT8) the conventional choice for biases?

---

## What you learned

Quantization in production frameworks (PyTorch FX, ONNX Runtime, TensorRT, llama.cpp) is doing **exactly this arithmetic**, just with more careful attention to:

- calibration (how do you pick `scale` for activations without a training run?),
- per-channel vs per-tensor (rows vs whole matrix),
- INT8 matmul kernels (GPU / CPU SIMD),
- dequantize points (end of each layer vs end of network).

But the math at the core is the same three lines you wrote above.
